In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType
from pyspark.sql import Row
catalog_name='ecommerce'

**Product Gold Table**

In [0]:
df_slv_product= spark.table(f"{catalog_name}.silver.slv_product")
df_slv_brands= spark.table(f"{catalog_name}.silver.slv_brands")
df_slv_category=spark.table(f"{catalog_name}.silver.slv_category") 

In [0]:
df_slv_product.createOrReplaceTempView("v_products")
df_slv_brands.createOrReplaceTempView("v_brands")
df_slv_category.createOrReplaceTempView("v_category")

In [0]:
spark.sql(f"USE CATALOG {catalog_name}") 

In [0]:
%sql
CREATE OR REPLACE TABLE gold.gld_dim_products AS

WITH brands_categories AS (
  SELECT
    b.brand_name,
    b.brand_code,
    c.category_name,
    c.category_code
  FROM v_brands b
  INNER JOIN v_category c
  ON
    b.category_code = c.category_code
)
SELECT
  p.product_id,
  p.sku,
  p.category_code,
  COALESCE(bc.category_name, 'Not Available') AS category_name,
  p.brand_code,
  COALESCE(bc.brand_name, 'Not Available')   AS brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.source_filename,
  p.created_at
FROM v_products p
LEFT JOIN brands_categories bc
  ON p.brand_code = bc.brand_code;

**Customer Gold Table**

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  

In [0]:
#Faltten the dictionary
rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
# 2️ Create mapping DataFrame
df_region_mapping = spark.createDataFrame(rows)

# Optional: show mapping
df_region_mapping.show(truncate=False)

In [0]:
df_slv_customer= spark.table(f"{catalog_name}.silver.slv_customer")
df_gold_customer=df_slv_customer.join(df_region_mapping,on=['country','state'],how="left")
df_gold_customer=df_gold_customer.fillna({"region":"Other"})
display(df_gold_customer.limit(5))

In [0]:
df_gold_customer.write\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")

**Calender Gold Table**

In [0]:
df_slv_calendar = spark.table(f"{catalog_name}.silver.slv_calendar")
df_gld_calendar = df_slv_calendar.withColumn("dateid",f.date_format(f.col("date"),"yyyyMMdd").cast("int"))\
    .withColumn("month_name",f.date_format(f.col("date"),"MMMM"))\
    .withColumn("is_weekend",f.when(f.col("day_name").isin(["Saturday","Sunday"]),1).otherwise(0))

display(df_gld_calendar.limit(5))



In [0]:
desired_columns_order = ["dateid", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "created_at", "source_filename"]

df_gld_calendar = df_gld_calendar.select(desired_columns_order)

display(df_gld_calendar.limit(5))

In [0]:
# write table to gold layer
df_gld_calendar.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")